In [1]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://ekitan.com/event/station-2590',
   'title': '東京駅周辺のイベント - 駅探',
   'content': '東京駅のイベント一覧 · 条件指定 · 京都アカデミアワークショップ2025 · 地域から日本の元気を考える 神戸豚まん企業の挑戦！ · ひろしま広域都市圏特産品フェアin東京 はっし',
   'score': 0.755078,
   'raw_content': None},
  {'url': 'https://www.walkerplus.com/event_list/ar0313/sc309880d/',
   'title': '東京駅(東京都)周辺のイベント - ウォーカープラス',
   'content': '東京駅(東京都)周辺で開催されるイベント情報29件をお届けします。今日開催されているイベントはもちろん、週末の「どこ行こう」に役立つ情報が満載！',
   'score': 0.7197191,
   'raw_content': None},
  {'url': 'https://www.enjoytokyo.jp/event/list/area1306/',
   'title': '東京駅周辺・丸の内でおすすめのイベント',
   'content': '東京駅・丸の内の開催中または開催予定のイベントを紹介。おでかけに詳しい編集部がセレクトしました。気になるイベントは開催期間や開催場所、最寄駅、地図、料金など',
   'score': 0.6882385,
   'raw_content': None},
  {'url': 'https://www.enjoytokyo.jp/event/list/sta200101/its04/',
   'title': '今日行ける！東京駅周辺のおすすめイベント',
   'content': '今日行ける！東京駅周辺のおすすめイベント ; WEEKLY PALETTE「赤福／五十鈴茶屋」. 2025/11/26(水) ～ 12/09(火) ; 丸の内イルミネーション 2025 ; MARUNOUCHI BRIGHT HOLIDAY',
   'score': 0.638588,


In [4]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [8]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。

- 東京都の面積は約2,194平方キロメートルです。
- 沖縄県の面積は約2,271平方キロメートルです。

したがって、沖縄県の方が東京都よりも広いです。


In [9]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近1ヶ月以内に東京駅周辺で行われるイベントに関する情報は以下の通りです。

1. **東京駅周辺イベント情報**
   - **イベント名**: 東京駅周辺のイベント
   - **開催期間**: 2025年11月13日（木）〜 2026年2月15日（日）
   - **詳細**: 各地でのクリスマスイベントや展示会が行われます。例えば、東京駅近くでのイルミネーションや特別な展示などがあります。  
   - **URL**: [Enjoy Tokyo](https://www.enjoytokyo.jp/event/list/area1306/)

2. **東京エキマチライブ**
   - **開催内容**: 音楽イベントや特別なパフォーマンスなどが定期的に行われます。詳細は公式サイトで確認できます。
   - **URL**: [Tokyo Station City](https://tokyostationcity-ekimachilive.com/)

これらのイベントは今後の季節に向けての準備や、特別な体験を提供するもので、多くの人々が楽しむことができる内容となっています。興味のある方は、各イベントの詳細を確認してみてください。


In [10]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは！'

こんにちは！何かお手伝いできることがありますか？


'質問:東北6県は？'

東北地方は、日本の地方の一つで、以下の6つの県から構成されています。

1. 青森県（あおもりけん）
2. 岩手県（いわてけん）
3. 宮城県（みやぎけん）
4. 秋田県（あきたけん）
5. 山形県（やまがたけん）
6. 福島県（ふくしまけん）

これらの県は、自然豊かで、観光地や文化が多様な地域です。


'質問:宮城県のお土産について検索した結果を教えて'

宮城県のおすすめお土産についての情報をいくつか紹介します。

1. **牛たん** - 宮城県名物の牛たんは、しっかりした食感と風味が特徴です。多くの店舗で販売されており、旅行の思い出として持ち帰るのもおすすめです。
   - 詳細: [Atlas Log](https://atlas-log.com/miyagi-souvenir/)

2. **ずんだ餅** - 枝豆を使った甘い餡で包まれた餅は、宮城の伝統的なスイーツです。お土産にぴったりです。
   - 詳細: [宮城県物産振興協会](https://ec.miyagibussan.or.jp/)

3. **萩の月** - しっとりとしたクリームが入ったカステラ生地のお菓子で、多くの観光客に人気です。
   - 詳細: [Jalan.net](https://www.jalan.net/omiyage/040000/)

4. **笹かまぼこ** - 新鮮な魚を使用した、独特の食感と風味が特徴のかまぼこです。手土産にも喜ばれます。
   - 詳細: [東北るるぶ](https://tohokuru.jp/blogs/feature/miyage_miyagi)

5. **蔵王チーズ** - 新鮮な牛乳を使用して作られるチーズ製品は、ワインとの相性が良く、おつまみとしても喜ばれます。
   - 詳細: [BIGLOBE Food](https://food.biglobe.ne.jp/prefectures_tohoku_miyagi/feature_longevity/)

これらのお土産は、宮城県の味を持ち帰るのに最適なアイテムです。興味がある方はぜひチェックしてみてください！

---ご利用ありがとうございました！---
